## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Conditional Generative Adversarial Networks (cGANs)
*****

__Basada en la implementación:__ [Simple Schwarz](https://medium.com/@simple.schwarz/how-to-build-a-cgan-for-generating-mnist-digits-in-pytorch-b74b59b77e7c
)

## Librerias

In [ ]:
import os
import sys
import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data
import torchvision.transforms as transforms
import torchvision.datasets as dset
import torchvision.utils as vutils

## Setup CUDA

In [ ]:
## Verificar acceso de uso a CUDA

CUDA = True
seed = 0

CUDA = CUDA and torch.cuda.is_available()
print("PyTorch version: {}".format(torch.__version__))
if CUDA:
    print("CUDA version: {}\n".format(torch.version.cuda))

if CUDA:
    torch.cuda.manual_seed(seed)
device = torch.device("cuda:0" if CUDA else "cpu")
cudnn.benchmark = True

## Diseño del modelo

In [ ]:
class Generator(nn.Module):
    def __init__(self, classes, channels, img_size, latent_dim):
        """
            DESCRIPTION:
                Generator model builder

            INPUT:
                @param classes: number of class in the dataset
                @type classes: int

                @param channels: number of channels of each image
                @type channels: int

                @param img_size: image size (for image dim: image size x image size)
                @type img_size: int

                @param latent_dim: latent or noise vector dimension
                @type latent_dim: int
        """
        super(Generator, self).__init__()
        self.classes = classes
        self.channels = channels
        self.img_size = img_size
        self.latent_dim = latent_dim
        self.img_shape = (self.channels, self.img_size, self.img_size)
        self.label_embedding = nn.Embedding(self.classes, self.classes)

        self.model = nn.Sequential(
            *self._create_layer(self.latent_dim + self.classes, 128, False),
            *self._create_layer(128, 256),
            *self._create_layer(256, 512),
            *self._create_layer(512, 1024),
            nn.Linear(1024, int(np.prod(self.img_shape))),
            nn.Tanh()
        )

    def _create_layer(self, size_in, size_out, normalize=True):
        layers = [nn.Linear(size_in, size_out)]
        if normalize:
            layers.append(nn.BatchNorm1d(size_out))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        return layers

    def forward(self, noise, labels):
        z = torch.cat((self.label_embedding(labels), noise), -1)
        x = self.model(z)
        x = x.view(x.size(0), *self.img_shape)
        return x


class Discriminator(nn.Module):
    def __init__(self, classes, channels, img_size, latent_dim):
        """
            DESCRIPTION:
                Discriminator model builder

            INPUT:
                @param classes: number of class in the dataset
                @type classes: int

                @param channels: number of channels of each image
                @type channels: int

                @param img_size: image size (for image dim: image size x image size)
                @type img_size: int

                @param latent_dim: latent or noise vector dimension
                @type latent_dim: int
        """
        super(Discriminator, self).__init__()
        self.classes = classes
        self.channels = channels
        self.img_size = img_size
        self.latent_dim = latent_dim
        self.img_shape = (self.channels, self.img_size, self.img_size)
        self.label_embedding = nn.Embedding(self.classes, self.classes)
        self.adv_loss = torch.nn.BCELoss()

        self.model = nn.Sequential(
            *self._create_layer(self.classes + int(np.prod(self.img_shape)), 1024, False, True),
            *self._create_layer(1024, 512, True, True),
            *self._create_layer(512, 256, True, True),
            *self._create_layer(256, 128, False, False),
            *self._create_layer(128, 1, False, False),
            nn.Sigmoid()
        )

    def _create_layer(self, size_in, size_out, drop_out=True, act_func=True):
        layers = [nn.Linear(size_in, size_out)]
        if drop_out:
            layers.append(nn.Dropout(0.4))
        if act_func:
            layers.append(nn.LeakyReLU(0.2, inplace=True))
        return layers

    def forward(self, image, labels):
        x = torch.cat((image.view(image.size(0), -1), self.label_embedding(labels)), -1)
        return self.model(x)

    def loss(self, output, label):
        return self.adv_loss(output, label)

## Dataset

<center>
    <img src=https://upload.wikimedia.org/wikipedia/commons/f/f7/MnistExamplesModified.png width=800>
</center>

MNIST (Modified National Institute of Standards and Technology) es una gran base de datos de dígitos escritos a mano que se utiliza habitualmente para entrenar diversos sistemas de procesamiento de imágenes. La base de datos contiene 60.000 imágenes de entrenamiento

**Objetivo**: Clasificar las imágenes según su dígito.


#### Carga de datos

In [ ]:
batch_size = 128
epochs = 5
lr = 2e-4
classes = 10
channels = 1
img_size = 64
latent_dim = 100
log_interval = 100

In [ ]:
## Descarga de la data y aplicacion de transformaciones
dataset = dset.MNIST(root='./data', download=True,
                     transform=transforms.Compose([
                     transforms.Resize(img_size),
                     transforms.ToTensor(),
                     transforms.Normalize((0.5,), (0.5,))
                     ]))

## Gestor de datos
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

## Modelo

In [ ]:
# Setup the generator and the discriminator
netG = Generator(classes, channels, img_size, latent_dim).to(device)
print(netG)

netD = Discriminator(classes, channels, img_size, latent_dim).to(device)
print(netD)

# Setup Adam optimizers for both G and D
optim_D = optim.Adam(netD.parameters(), lr=lr, betas=(0.5, 0.999))
optim_G = optim.Adam(netG.parameters(), lr=lr, betas=(0.5, 0.999))

In [ ]:
## Entrenamiento del modelo
img_list = []

netG.train()
netD.train()
viz_z = torch.zeros((batch_size, latent_dim), device=device)
viz_noise = torch.randn(batch_size, latent_dim, device=device)
nrows = batch_size // 8
viz_label = torch.LongTensor(np.array([num for _ in range(nrows) for num in range(8)])).to(device)

for epoch in range(epochs):
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        batch_size = data.size(0)
        real_label = torch.full((batch_size, 1), 1., device=device)
        fake_label = torch.full((batch_size, 1), 0., device=device)

        # Training generator
        netG.zero_grad()
        z_noise = torch.randn(batch_size, latent_dim, device=device)
        x_fake_labels = torch.randint(0, classes, (batch_size,), device=device)
        x_fake = netG(z_noise, x_fake_labels)
        y_fake_g = netD(x_fake, x_fake_labels)
        g_loss = netD.loss(y_fake_g, real_label)
        g_loss.backward()
        optim_G.step()

        # Training Discriminator
        netD.zero_grad()
        y_real = netD(data, target)
        d_real_loss = netD.loss(y_real, real_label)
        y_fake_d = netD(x_fake.detach(), x_fake_labels)
        d_fake_loss = netD.loss(y_fake_d, fake_label)
        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optim_D.step()

        if batch_idx % log_interval == 0 and batch_idx > 0:
            print('Epoch {} [{}/{}] loss_D: {:.4f} loss_G: {:.4f}'.format(
                        epoch, batch_idx, len(dataloader),
                        d_loss.mean().item(),
                        g_loss.mean().item()))

            with torch.no_grad():
                viz_sample = netG(viz_noise, viz_label)
                img_list.append(vutils.make_grid(viz_sample, normalize=True))


In [ ]:
# Plot the fake images
plt.figure(figsize=(15,15))
plt.axis("off")
plt.title("Fake Images")
plt.imshow(np.transpose(img_list[-1],(1,2,0)))
plt.show()


#### Generar imagenes

In [ ]:
## Set model to evaluate mode
netG.eval()
y_test = [0,0,1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9]
ba_size = len(y_test)

## Compute prediction
with torch.no_grad():
    z_noise = torch.randn(ba_size, latent_dim, device=device)
    x_fake_labels = torch.IntTensor(y_test, device=device)
    x_fake = netG(z_noise, x_fake_labels)


In [ ]:
# Plot the fake images
imgs = vutils.make_grid(x_fake, normalize=True)

plt.figure(figsize=(15,15))
plt.axis("off")
plt.title("Fake Images")
plt.imshow(np.transpose(imgs,(1,2,0)))
plt.show()
